# DSN BOOTCAMP QUALIFIER PROJECT 2026.

## 1. Importation of libraries.

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as pyplot
import sklearn
%matplotlib inline

In [2]:
train_df = pd.read_csv('train.csv')
test_df=pd.read_csv('test.csv')

In [3]:
test_df.head()

,id,product_code,product_weight_kg,fat_content,shelf_visibility,product_category,product_price,store_code,store_age_years,store_size,store_location_tier,store_format
0,row_00009,PRD-2WLNC8,8.628,Low Fat,0.0167,household,190.72,STORE-HL7,23,Medium,Tier_3,Superstore
1,row_00015,PRD-LV9PLM,6.993,Low Fat,0.0579,Health and Hygiene,261.07,STORE-9RG,28,Small,Tier_2,Standard Supermarket
2,row_00019,PRD-ET6ZI7,NaN,Low Fat,0.1262,FRUITS AND VEGETABLES,112.50,STORE-7WS,47,Medium,Tier_3,Flagship Hypermarket
3,row_00020,PRD-6RX35U,14.034,Regular,0.0761,frozen foods,200.71,STORE-DKU,30,NaN,Tier_2,Standard Supermarket
4,row_00023,PRD-I4J38V,13.063,Low Fat,0.0650,Frozen Foods,150.59,STORE-89Z,33,Medium,Tier_1,Standard Supermarket


In [4]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1705 entries, 0 to 1704
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   1705 non-null   object 
 1   product_code         1705 non-null   object 
 2   product_weight_kg    1399 non-null   float64
 3   fat_content          1705 non-null   object 
 4   shelf_visibility     1705 non-null   float64
 5   product_category     1705 non-null   object 
 6   product_price        1705 non-null   float64
 7   store_code           1705 non-null   object 
 8   store_age_years      1705 non-null   int64  
 9   store_size           1214 non-null   object 
 10  store_location_tier  1705 non-null   object 
 11  store_format         1705 non-null   object 
dtypes: float64(3), int64(1), object(8)
memory usage: 160.0+ KB


In [5]:
test_df.describe()

,product_weight_kg,shelf_visibility,product_price,store_age_years
count,1399.000000,1705.000000,1705.000000,1705.000000
mean,12.865960,0.068491,143.069490,34.127859
std,4.647713,0.051721,61.888692,8.322650
min,4.693000,0.000000,31.980000,23.000000
25%,8.753000,0.030200,96.560000,28.000000
50%,12.617000,0.056400,144.630000,33.000000
75%,16.915500,0.098400,185.210000,45.000000
max,21.980000,0.302000,270.870000,47.000000


In [6]:
test_df.isnull().sum()

id                       0
product_code             0
product_weight_kg      306
fat_content              0
shelf_visibility         0
product_category         0
product_price            0
store_code               0
store_age_years          0
store_size             491
store_location_tier      0
store_format             0
dtype: int64

# 2. Data Cleaning

In [7]:
# now, let's normalize category casing on both test data like it was on train data.
train_df['product_category'] = train_df['product_category'].str.strip().str.lower()
test_df['product_category'] = test_df['product_category'].str.strip().str.lower()

In [8]:
# category lookups from the train notebook.
weight_by_code = train_df.groupby('product_code')['product_weight_kg'].median()
weight_by_category = train_df.groupby('product_category')['product_weight_kg'].median()
weight_global_median = train_df['product_weight_kg'].median()

In [9]:
def fill_weight(row):
    if pd.notna(row['product_weight_kg']):
        return row['product_weight_kg']
    if row['product_code'] in weight_by_code.index and pd.notna(weight_by_code[row['product_code']]):
        return weight_by_code[row['product_code']]
    if row['product_category'] in weight_by_category.index:
        return weight_by_category[row['product_category']]
    return weight_global_median

In [10]:
test_df['product_weight_kg'] = test_df.apply(fill_weight, axis=1)

In [11]:
known = train_df.dropna(subset=['store_size'])
combo_mode = known.groupby(['store_format', 'store_location_tier'])['store_size'].agg(lambda x: x.mode()[0])
format_mode = known.groupby('store_format')['store_size'].agg(lambda x: x.mode()[0])
global_mode = known['store_size'].mode()[0]

In [12]:
def fill_store_size(row):
    if pd.notna(row['store_size']):
        return row['store_size']
    key = (row['store_format'], row['store_location_tier'])
    if key in combo_mode.index:
        return combo_mode[key]
    if row['store_format'] in format_mode.index:
        return format_mode[row['store_format']]
    return global_mode

In [13]:
test_df['store_size'] = test_df.apply(fill_store_size, axis=1)

In [14]:
test_df.isnull().sum()

id                     0
product_code           0
product_weight_kg      0
fat_content            0
shelf_visibility       0
product_category       0
product_price          0
store_code             0
store_age_years        0
store_size             0
store_location_tier    0
store_format           0
dtype: int64

In [15]:
test_df

,id,product_code,product_weight_kg,fat_content,shelf_visibility,product_category,product_price,store_code,store_age_years,store_size,store_location_tier,store_format
0,row_00009,PRD-2WLNC8,8.628,Low Fat,0.0167,household,190.72,STORE-HL7,23,Medium,Tier_3,Superstore
1,row_00015,PRD-LV9PLM,6.993,Low Fat,0.0579,health and hygiene,261.07,STORE-9RG,28,Small,Tier_2,Standard Supermarket
2,row_00019,PRD-ET6ZI7,17.328,Low Fat,0.1262,fruits and vegetables,112.50,STORE-7WS,47,Medium,Tier_3,Flagship Hypermarket
3,row_00020,PRD-6RX35U,14.034,Regular,0.0761,frozen foods,200.71,STORE-DKU,30,Small,Tier_2,Standard Supermarket
4,row_00023,PRD-I4J38V,13.063,Low Fat,0.0650,frozen foods,150.59,STORE-89Z,33,Medium,Tier_1,Standard Supermarket
...,...,...,...,...,...,...,...,...,...,...,...,...
1700,row_08483,PRD-58L9K5,17.919,Low Fat,0.0607,starchy foods,168.62,STORE-89Z,33,Medium,Tier_1,Standard Supermarket
1701,row_08489,PRD-0B58ST,16.748,Regular,0.0136,canned,97.26,STORE-JOR,34,Small,Tier_3,Corner Shop
1702,row_08491,PRD-RH8H90,9.876,Low Fat,0.0239,health and hygiene,183.35,STORE-HL7,23,Medium,Tier_3,Superstore
1703,row_08504,PRD-J51IFI,12.120,Low Fat,0.0221,breads,166.09,STORE-7WS,47,Medium,Tier_3,Flagship Hypermarket


# 3. Ordinal Encoding for hierachical datas

In [16]:
size_order = ['Small', 'Medium', 'Large']
tier_order = ['Tier_1', 'Tier_2', 'Tier_3']

In [17]:
for df in [train_df, test_df]:
    df['store_size'] = pd.Categorical(df['store_size'], categories=size_order, ordered=True).codes
    df['store_location_tier'] = pd.Categorical(df['store_location_tier'], categories=tier_order, ordered=True).codes

In [18]:
# Encoding both train and test data to have a matching result.
cat_cols = ['fat_content', 'store_format', 'product_category', 'store_code']
combined = pd.concat([train_df[cat_cols], test_df[cat_cols]], keys=['train', 'test'])
combined_encoded = pd.get_dummies(combined, columns=cat_cols, drop_first=True)

In [19]:
train_encoded = combined_encoded.loc['train'].reset_index(drop=True)
test_encoded = combined_encoded.loc['test'].reset_index(drop=True)

In [20]:
test_df = pd.concat([test_df.drop(columns=cat_cols).reset_index(drop=True), test_encoded], axis=1)

In [21]:
test_df = test_df.drop(columns=['product_code'])

In [22]:
test_df.shape

(1705, 35)

In [23]:
test_df

,id,product_weight_kg,shelf_visibility,product_price,store_age_years,store_size,store_location_tier,fat_content_Regular,store_format_Flagship Hypermarket,store_format_Standard Supermarket,...,product_category_starchy foods,store_code_STORE-89Z,store_code_STORE-9RG,store_code_STORE-AGY,store_code_STORE-DKU,store_code_STORE-HL7,store_code_STORE-JOR,store_code_STORE-OYG,store_code_STORE-T5G,store_code_STORE-YLW
0,row_00009,8.628,0.0167,190.72,23,1,2,False,False,False,...,False,False,False,False,False,True,False,False,False,False
1,row_00015,6.993,0.0579,261.07,28,0,1,False,False,True,...,False,False,True,False,False,False,False,False,False,False
2,row_00019,17.328,0.1262,112.50,47,1,2,False,True,False,...,False,False,False,False,False,False,False,False,False,False
3,row_00020,14.034,0.0761,200.71,30,0,1,True,False,True,...,False,False,False,False,True,False,False,False,False,False
4,row_00023,13.063,0.0650,150.59,33,1,0,False,False,True,...,False,True,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1700,row_08483,17.919,0.0607,168.62,33,1,0,False,False,True,...,True,True,False,False,False,False,False,False,False,False
1701,row_08489,16.748,0.0136,97.26,34,0,2,True,False,False,...,False,False,False,False,False,False,True,False,False,False
1702,row_08491,9.876,0.0239,183.35,23,1,2,False,False,False,...,False,False,False,False,False,True,False,False,False,False
1703,row_08504,12.120,0.0221,166.09,47,1,2,False,True,False,...,False,False,False,False,False,False,False,False,False,False
